# Cuisine Classification with Machine Learning
Predicting cuisine type based on ingredient presence

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('cuisines.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.cuisine.value_counts().plot(kind='barh')
plt.title('Cuisine Distribution')
plt.show()

## Separate Data by Cuisine Type

In [ ]:
thai_df = df[(df.cuisine == 'thai')]
japanese_df = df[(df.cuisine == 'japanese')]
chinese_df = df[(df.cuisine == 'chinese')]
indian_df = df[(df.cuisine == 'indian')]
korean_df = df[(df.cuisine == 'korean')]

print(f'Thai df : {thai_df.shape}')
print(f'Japanese df : {japanese_df.shape}')
print(f'Chinese df : {chinese_df.shape}')
print(f'Indian df : {indian_df.shape}')
print(f'Korean df : {korean_df.shape}')

## Top 10 Ingredients by Cuisine

In [ ]:
def create_ingredient_df(df):
    ingredient_df = df.T.drop(['cuisine', 'Unnamed: 0']).sum(axis=1).to_frame('value')
    ingredient_df = ingredient_df[(ingredient_df.T != 0).any()]
    ingredient_df = ingredient_df.sort_values(by='value', ascending=False, inplace=False)
    return ingredient_df

In [ ]:
thai_ingredient_df = create_ingredient_df(thai_df)
thai_ingredient_df.head(10).plot.barh()
plt.title('Top 10 Thai Ingredients')
plt.show()

In [ ]:
japanese_ingredient_df = create_ingredient_df(japanese_df)
japanese_ingredient_df.head(10).plot.barh()
plt.title('Top 10 Japanese Ingredients')
plt.show()

In [ ]:
chinese_ingredient_df = create_ingredient_df(chinese_df)
chinese_ingredient_df.head(10).plot.barh()
plt.title('Top 10 Chinese Ingredients')
plt.show()

In [ ]:
indian_ingredient_df = create_ingredient_df(indian_df)
indian_ingredient_df.head(10).plot.barh()
plt.title('Top 10 Indian Ingredients')
plt.show()

In [ ]:
korean_ingredient_df = create_ingredient_df(korean_df)
korean_ingredient_df.head(10).plot.barh()
plt.title('Top 10 Korean Ingredients')
plt.show()

## Prepare Features and Labels

In [ ]:
feature_df = df.drop(['cuisine', 'Unnamed: 0', 'rice', 'garlic', 'ginger'], axis=1)
labels_df = df.cuisine
feature_df.head()

## Handle Class Imbalance with SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

oversample = SMOTE()
transformed_feature_df, transformed_label_df = oversample.fit_resample(feature_df, labels_df)
print(f'New label count:\n{transformed_label_df.value_counts()}')
print(f'\nOld label count:\n{labels_df.value_counts()}')

## Train Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    transformed_feature_df, transformed_label_df, test_size=0.3, random_state=42
)

lr = LogisticRegression(multi_class='ovr', solver='liblinear', max_iter=1000)
model = lr.fit(X_train, np.ravel(y_train))

accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy:.4f}")

## Make a Prediction on Test Sample

In [ ]:
# Ingredients used in sample
sample_index = 50
print(f'Ingredients: {X_test.iloc[sample_index][X_test.iloc[sample_index] != 0].index.tolist()}')
print(f'Actual cuisine: {y_test.iloc[sample_index]}')

In [ ]:
# Get prediction probabilities
test = X_test.iloc[sample_index].values.reshape(1, -1)
proba = model.predict_proba(test)
classes = model.classes_
resultdf = pd.DataFrame(data=proba, columns=classes)

# Top predictions
toppred = resultdf.T.sort_values(by=[0], ascending=[False])
print('\nPrediction Probabilities (sorted):')
print(toppred)

## Model Evaluation Report

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))